# 08 · Profiling & memory

Pairs with `docs/04` M7-M9 and `docs/05-profiling.md`. Allocator behavior, sync
traps, and reading a profiler trace. GPU cells guarded for CPU runs.

In [ ]:
import torch
from gpulab.learn import inspect as I
HAS_CUDA = torch.cuda.is_available()
dev = torch.device("cuda" if HAS_CUDA else "cpu")
print("device:", dev, "| CUDA:", HAS_CUDA)
if not HAS_CUDA:
    print("No CUDA here - cells run on CPU; the timing/memory numbers are only")
    print("meaningful on your RTX 3060. Run this notebook there for the real story.")

## M7 — allocated vs reserved (the caching allocator)

In [ ]:
if HAS_CUDA:
    I.reset_cuda_peak(); I.cuda_mem("start")
    a = torch.empty(256, 1024, 1024, device=dev)   # 1 GB
    I.cuda_mem("after 1GB alloc")
    del a
    I.cuda_mem("after del (still reserved!)")
    torch.cuda.empty_cache()
    I.cuda_mem("after empty_cache (returned to driver)")
else:
    print("GPU-only: `reserved` > `allocated` because PyTorch caches freed blocks.")

In [ ]:
# Optional (GPU): record a memory history and open it at https://pytorch.org/memory_viz
# torch.cuda.memory._record_memory_history()
# ... run a training step ...
# torch.cuda.memory._dump_snapshot("mem.pickle")

## M9 — synchronization traps

In [ ]:
import numpy as np, time
Xg = torch.tensor(np.random.default_rng(0).standard_normal((8192, 120)).astype("float32")).to(dev)
from gpulab.models.cnn1d import CNN1D
model = CNN1D(3, 120).to(dev)

def loop(sync_every_step):
    for i in range(0, 8192, 512):
        out = model(Xg[i:i+512]).sum()
        if sync_every_step:
            _ = out.item()          # forces a CPU<->GPU sync every step
    if HAS_CUDA: torch.cuda.synchronize()

for flag in (False, True):
    t0 = time.perf_counter(); loop(flag); dt = time.perf_counter() - t0
    print(f"sync_every_step={flag}: {dt*1e3:.1f} ms")
print("On GPU the sync-every-step version is markedly slower (that's the trap).")

## Profiler — which kernels cost the most

In [ ]:
I.profile(lambda: model(Xg[:1024]).sum().backward(), row_limit=12)
# Add trace_path='trace.json' and open in chrome://tracing / ui.perfetto.dev.

> **Concepts to note** (copy into your own theory notebook):
> - `reserved` > `allocated`: PyTorch caches freed blocks (cudaMalloc is slow).
> - `empty_cache()` returns memory to the driver and usually makes you SLOWER.
> - Fragmentation -> OOM with free memory available.
> - `.item()`/print/`.cpu()` in a hot loop serialize CPU<->GPU (a top perf bug).
> - Profile STEADY STATE (skip warmup); always synchronize before a CPU timer.